# 00. 環境チェック

このノートブックが最後まで緑になるまで、他のノートブックに進まないこと。
ここで落ちる項目は、そのまま Web アプリでも落ちる。

確認するのは 5 つ:

1. Ollama に到達できるか / 使うモデルが pull 済みか
2. リソースポリシー（商用限定）が読めるか
3. biomni がインストールされているか
4. **biomni の `default_config` が Ollama を向いているか**（docs/design/04 §4.3）
5. データレイクに必要なファイルがあるか

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings, apply_biomni_env
from biomni_hypo.llm import ollama_status
from biomni_hypo.policy import ResourcePolicy

settings = Settings()
print("model            :", settings.model)
print("ollama_base_url  :", settings.ollama_base_url)
print("num_ctx          :", settings.num_ctx)
print("data_path        :", settings.data_path)
print("use_tool_retriever:", settings.use_tool_retriever)
print("commercial_mode  :", settings.commercial_mode)

## 1. Ollama

`reachable=False` なら `ollama serve` が動いていない。
使うモデルが一覧に無ければ `ollama pull qwen3:14b`。

In [ ]:
status = ollama_status(settings.ollama_base_url)
print("reachable:", status.reachable, status.error[:120])

## 2. リソースポリシー（商用限定）

`A1(commercial_mode=True)` はデータセットしか絞らない。ツールは絞らないので、
このポリシーが商用利用の実質的な担保になる（docs/design/05）。
モデルのライセンス判定もここが持つ。

In [ ]:
policy = ResourcePolicy.load(settings.policy_path)
print("policy version :", policy.version, "/ mode:", policy.mode)
print("許可データセット:", len(policy.allowed_dataset_names()))
print("拒否ツール      :", policy.denied_tool_names())
print("推奨モデル      :", policy.allowed_model_names())

## 3. ローカルにあるモデルを読み込んで選ぶ

`ollama pull` したモデルを読み込み、商用利用ポリシーで判定する。

- **★ 推奨 / ✓ 選択可**: そのまま使える
- **✕**: ローカルにはあるがライセンスが通らない（理由を出す。黙って隠さない）
- **…**: 未取得。`ollama pull <名前>` で入る

タグ違い（`qwen3:8b-instruct-q4_K_M` など）もファミリー名で判定するので、
許可リストに無い名前でも正しく拾える。

In [ ]:
from biomni_hypo.models import list_local_models

catalog = list_local_models(settings, policy)
print(catalog.as_table())

In [ ]:
from biomni_hypo.models import ModelNotAvailable, apply_model_selection

# 使いたいモデルをここで指定する。None なら settings.model -> カタログの既定 の順で決まる
CHOICE = None    # 例: "qwen3:14b"

try:
    _catalog, notes = apply_model_selection(settings, policy, model=CHOICE)
    for note in notes:
        print("⚠️ ", note)
    print(f"\n使用モデル: {settings.model}")
    print(f"num_ctx   : {settings.num_ctx:,}  (モデル上限に合わせて丸め済み)")
except ModelNotAvailable as exc:
    print("❌", exc)

`apply_model_selection()` は **Web アプリの `POST /api/runs` が呼ぶのと同じ関数**。
ここで選べたモデルは API でもそのまま選べる。

`.env` の既定を変えるなら:

```bash
python scripts/list_models.py                 # 一覧
python scripts/list_models.py --set qwen3:8b  # HYPO_MODEL と BIOMNI_LLM を書き換える
```

`BIOMNI_LLM` も一緒に変える必要がある（biomni の DB クエリツールは
A1 のコンストラクタ引数ではなく `default_config` を見るため / §4.3）。スクリプトは両方直す。

## 4-5. 依存パッケージと biomni の default_config

`import biomni` が通るだけでは足りない。**biomni 0.0.8 の `pyproject.toml` は
`pydantic` / `langchain` / `python-dotenv` しか宣言しておらず**、`pandas` と
`langchain-openai` が無いと `from biomni.agent import A1` の時点で落ちる。
Ollama を使うには `langchain-ollama` も要る（これが無いと 01 で初めて失敗する）。

ここで必要なものを全部確認しておく。

そのうえで **順序が重要**: `apply_biomni_env()` を biomni の import より前に呼ぶ。
`biomni.config.default_config` はモジュール読み込み時に環境変数を読むため、
後から設定しても `biomni/tool/database.py` が Anthropic を呼びに行く。

モデルを選んだ *あと* に呼ぶこと（`BIOMNI_LLM` に選択したモデル名が入る）。

In [ ]:
applied = apply_biomni_env(settings)   # ← import より前
for k, v in applied.items():
    print(f"{k:24s} = {v}")

In [ ]:
from biomni_hypo.config import AGENT_DEPENDENCIES, install_hint, missing_dependencies

for dep in AGENT_DEPENDENCIES:
    print(("✅" if dep.installed else "❌"), f"{dep.module:20s} {dep.package:22s} {dep.why}")

missing = missing_dependencies()
DEPS_OK = not missing

if missing:
    print("\n❌ 足りない依存があります。これを実行してから続けてください:\n")
    print("   " + install_hint(missing))
    print("\n   （まとめて入れるなら: pip install -r requirements.txt）")
else:
    from biomni.version import __version__ as biomni_version
    print(f"\n✅ 依存は揃っています。biomni {biomni_version}")

In [ ]:
from biomni_hypo.config import assert_biomni_env

if DEPS_OK:
    assert_biomni_env(settings)     # 落ちたら §4.3 の環境変数設定を見直す
    print("✅ default_config は Ollama / 商用モードを向いています")

    from biomni.config import default_config
    print("   llm            :", default_config.llm)
    print("   source         :", default_config.source)
    print("   commercial_mode:", default_config.commercial_mode)

## 6. データレイク

`A1.__init__` は `expected_data_lake_files` を渡さないと**全ファイルを S3 から取得しようとする**
（数十 GB / docs/design/04 §4.4）。本アプリは許可リストのファイルだけを扱う。

In [ ]:
import pathlib

data_lake = pathlib.Path(settings.data_path) / "biomni_data" / "data_lake"
present = {p.name for p in data_lake.glob("*")} if data_lake.exists() else set()
allowed = policy.allowed_dataset_names()

print(f"data_lake: {data_lake}")
print(f"許可 {len(allowed)} 件中 {len(present & set(allowed))} 件が取得済み\n")
for name in allowed:
    mark = "✓" if name in present else "·"
    d = policy.check_dataset(name)
    flag = " ⚠️要ライセンス確認" if d.review_required else ""
    print(f"  {mark} {name:52s} {d.license}{flag}")

### 取得するには

```bash
python scripts/fetch_datasets.py            # 許可リスト全件
python scripts/fetch_datasets.py --only gwas_catalog.pkl gene_info.parquet
```

データが 0 件でも、ツール（公共 DB への問い合わせ）だけで動くランは実行できる。
まずは `gwas_catalog.pkl` と `gene_info.parquet` だけあれば 02 以降が試せる。

## チェック結果まとめ

In [ ]:
selectable = [x.name for x in catalog.selectable]
checks = {
    "Ollama 到達": status.reachable,
    "使えるモデルが 1 つ以上ある": bool(selectable),
    f"選択したモデル ({settings.model}) が使える": settings.model in selectable,
    "ポリシー読み込み": policy.version >= 1,
    "依存パッケージ": DEPS_OK,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

if not all(checks.values()):
    print("\n❌ の項目を解消してから 01 に進むこと。")
else:
    print("\n➡️  01_ollama_stop_sequence.ipynb へ")